# Initialize Data

In [1]:
# @title Mount and Import { display-mode: "form" }

%%time

import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

from IPython.display import clear_output

from google.colab import drive
import tensorflow as tf

from sqlalchemy import create_engine
import pandas as pd
import numpy as np

from tqdm.notebook import tqdm
from sklearn.preprocessing import MultiLabelBinarizer
from scipy.sparse import csr_matrix

def check_drive_mounted():
  return os.path.exists('/content/drive')

def check_gpu_available():
  return bool(tf.test.gpu_device_name())

def mount_google_drive():
  drive.mount('/content/drive')

def install_general_dependencies():
  !pip install isodate

def install_gpu_dependencies():
  !pip uninstall torchvision torchaudio torchtext -y
  !pip install --upgrade autogluon datasets

def clear_output_and_print_status():
  clear_output()
  print('Google Drive Mounted:', check_drive_mounted())
  print('GPU Availability:', check_gpu_available())
  print()

# Mount Google Drive if not already mounted
if not check_drive_mounted():
  mount_google_drive()

# Install dependencies for processor
install_general_dependencies()
if check_gpu_available():
  install_gpu_dependencies()

Mounted at /content/drive
Found existing installation: torchvision 0.21.0+cu124
Uninstalling torchvision-0.21.0+cu124:
  Successfully uninstalled torchvision-0.21.0+cu124
Found existing installation: torchaudio 2.6.0+cu124
Uninstalling torchaudio-2.6.0+cu124:
  Successfully uninstalled torchaudio-2.6.0+cu124
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 4.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.5/259.5 kB 16.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
INFO: pip is looking at multiple versions of openxlab to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of openxlab to determine which version is compatible 

CPU times: user 7.1 s, sys: 1.25 s, total: 8.35 s
Wall time: 3min 19s


In [2]:
# @title Read Data Provided

%%time

db_names = [
    # 'nfs.db',
    'nfs_m.db',
    # 'video_statistics.csv'
    ]

dfs = []

for db_name in db_names:
  if '.db' in db_name:
    db_loc = f'sqlite:///drive/MyDrive/colab/banshee/data/{db_name}'
    engine = create_engine(db_loc)
    dfs.append(pd.read_sql_table('video_statistics', engine))
  elif '.csv' in db_name:
    dfs.append(pd.read_csv(f'drive/MyDrive/colab/banshee/data/{db_name}',
                           low_memory=False))
  dfs[-1]['source'] = db_name
  print(f'Loaded {db_name}')

Loaded nfs_m.db
CPU times: user 16.9 s, sys: 3.89 s, total: 20.8 s
Wall time: 53.4 s


In [3]:
# @title Combine and Initial Preprocess

%%time

vid_stats_df = pd.concat(dfs)

import pandas as pd
from tqdm.notebook import tqdm

tqdm.pandas()

# Standardize Duration
import isodate

def parse_duration(value):
    # If the value is an integer, return it directly
    if isinstance(value, int):
        return value
    # If the value is a string
    elif isinstance(value, str):
        # If it's a string representation of an integer, convert to int
        if value.isdigit():
            return int(value)
        # If it starts with 'PT', parse it as an ISO 8601 duration
        elif value.startswith('PT'):
            try:
                duration = isodate.parse_duration(value)
                # Convert duration to total seconds
                total_seconds = int(duration.total_seconds())
                return total_seconds
            except Exception as e:
                print(f"Error parsing duration '{value}': {e}")
                return None
        else:
            # Handle any other unexpected string formats
            print(f"Unrecognized duration format: '{value}'")
            return None
    else:
        # Handle any other data types (e.g., floats, None)
        print(f"Unsupported data type: {type(value)}")
        return None

vid_stats_df['video_duration'] = vid_stats_df['video_duration'].progress_apply(parse_duration)

# Standardize Captions
def convert_caption(value):
    truthy_values = {1, '1', True, 'True', 'true'}
    falsy_values = {0, '0', False, 'False', 'false'}

    if value in truthy_values:
        return 1
    elif value in falsy_values:
        return 0
    else:
        # Handle unexpected values
        return np.nan  # You can choose to return 0, 1, or raise an error

vid_stats_df['video_caption'] = vid_stats_df['video_caption'].progress_apply(convert_caption)

import numpy as np

# Standardize Tags
vid_stats_df['video_tags'] = vid_stats_df['video_tags'].replace(['None', 'NaN', np.nan], '')

# Standardize Topic Categories
vid_stats_df['video_topic_categories'] = vid_stats_df['video_topic_categories'].replace(['None', 'NaN', np.nan], '')
vid_stats_df['video_topic_categories'] = vid_stats_df['video_topic_categories'].progress_apply(lambda x: x.replace(', ', ','))

# Standardize Published Datetime
vid_stats_df['video_published_at'] = pd.to_datetime(vid_stats_df['video_published_at'])
vid_stats_df['video_published_at'] = vid_stats_df['video_published_at'].dt.tz_localize(None)
vid_stats_df['video_published_at'] = vid_stats_df['video_published_at']

# Convert columnts to int
int_cols = ['video_published_at', 'video_duration', 'video_caption',
            'video_licensed_content', 'video_view_count', 'video_like_count',
            'video_comment_count']
vid_stats_df.dropna(subset=int_cols, inplace=True)

from pandas.errors import IntCastingNaNError

for col in int_cols:
  try:
    vid_stats_df[col] = vid_stats_df[col].astype(int)
  except IntCastingNaNError:
    vid_stats_df[col] = vid_stats_df[col].astype('Int64')

# Cleanup
vid_stats_df.sort_values('video_view_count', inplace=True)
vid_stats_df.drop_duplicates(subset=['video_id'], keep='last', inplace=True)
vid_stats_df.set_index('video_id', inplace=True)

vid_stats_df.shape

  0%|          | 0/1150532 [00:00<?, ?it/s]

  0%|          | 0/1150532 [00:00<?, ?it/s]

  0%|          | 0/1150532 [00:00<?, ?it/s]

CPU times: user 14.3 s, sys: 359 ms, total: 14.6 s
Wall time: 23.9 s


(1133416, 14)

In [4]:
# @title Create Topics DataFrame

%%time

from sklearn.preprocessing import MultiLabelBinarizer
from scipy.sparse import csr_matrix
import pandas as pd
import numpy as np

def create_topics_dataframe(vid_stats_df):
    # Step 1: Split and strip video topics (avoid extra lambda function if possible)
    video_topics = vid_stats_df['video_topic_categories'].str.split(',').apply(
        lambda x: [i.strip() for i in x] if isinstance(x, list) else [])

    # Step 2: MultiLabelBinarizer to convert topics to one-hot encoding
    mlb = MultiLabelBinarizer()
    binarized_video_topics = mlb.fit_transform(video_topics)

    # Step 3: Convert to a sparse DataFrame using CSR (Compressed Sparse Row) format
    topics_sparse_matrix = csr_matrix(binarized_video_topics)

    # Step 4: Perform element-wise multiplication with view counts directly
    view_counts = vid_stats_df['video_view_count'].astype(int).to_numpy()
    result_array = topics_sparse_matrix.multiply(view_counts[:, np.newaxis])

    # Step 5: Create a DataFrame from the result array and merge additional columns
    topics_df = pd.DataFrame.sparse.from_spmatrix(result_array, columns=mlb.classes_, index=vid_stats_df.index)

    # Step 6: Merge back 'channel_id', 'views', and 'video_published_at'
    topics_df = topics_df.merge(vid_stats_df[['channel_id', 'video_view_count', 'video_published_at']],
                                left_index=True, right_index=True)

    # Step 7: Drop 'nan' column if it exists
    if 'nan' in topics_df.columns:
        topics_df.drop(columns=['nan'], inplace=True)

    return topics_df

# Example of calling the function
topics_df = create_topics_dataframe(vid_stats_df)

topics_df.shape

CPU times: user 9.09 s, sys: 999 ms, total: 10.1 s
Wall time: 13.3 s


(1133416, 63)

In [5]:
# @title Post Niche DataFrame

%%time

niche_data = []
eval_current_data = []

# Group by 'channel_id'
groups = topics_df.groupby('channel_id')

for group_name, group_data in tqdm(groups):
    if len(group_data) > 1:
        # Sort data by timestamp
        group_data = group_data.sort_values('video_published_at')

        # Convert relevant columns to dense before cumsum
        group_dense_data = group_data.drop(['channel_id', 'video_published_at'], axis=1).apply(lambda col: col.to_numpy())

        # Cumulative sum of relevant columns
        group_cum_sum = group_dense_data.cumsum()

        # Calculate cumulative share
        group_share = group_cum_sum.div(group_cum_sum['video_view_count'], axis=0)

        # Cumulative count of videos
        group_share['video_count'] = np.arange(1, len(group_share) + 1)

        # Calculate cumulative mean and median using expanding window functions
        group_share['mean_views'] = group_data['video_view_count'].expanding().mean()
        group_share['median_views'] = group_data['video_view_count'].expanding().median()

        group_share['channel_id'] = group_name

        niche_data.append(group_share)

# Concatenate all group results into a single DataFrame
post_niche_df = pd.concat(niche_data)

post_niche_df.shape

  0%|          | 0/1643 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [6]:
# @title Create Niche DataFrame and Current Niche DataFrame

%%time

groups = post_niche_df.groupby('channel_id')

current_df = pd.concat([group_data.tail(1) for group_name, group_data
                          in tqdm(groups)]).set_index('channel_id')

niche_df = pd.concat([group_data.shift(1).dropna() for group_name, group_data
                          in tqdm(groups)]).drop(['channel_id',
                                                  'video_view_count'], axis=1)

full_df = vid_stats_df.merge(niche_df, left_index=True, right_index=True)

full_df.shape

NameError: name 'post_niche_df' is not defined

* Should mean/median be assigned per time of upload, or now?
* Still need to try encodings over categories
* Restoring durations and numbers


In [7]:
# @title Data Parameters

import ipywidgets as widgets
from IPython.display import display, HTML
import random

from IPython.display import HTML

# Set Variables
threshold_value = 100000
segment_value = 'Racing Channels'
video_value = 'All'


custom_css = HTML("""
<style>
    .widget-label {
        font-size: 16px !important;  /* Increase font size for labels */
    }
    .widget-slider .widget-readout {
        font-size: 16px !important;  /* Increase font size for slider readout */
    }
</style>
""")

display(custom_css)

# Target Binary Threshold
target_style = widgets.ToggleButtons(
    options=['Median', 'Mean'],
    description='Target Binary Threshold',
    style={'description_width': 'initial'}
)

# Resample Mode
resample_mode = widgets.ToggleButtons(
    options=[None, True, False],
    description='Resample Mode',
    style={'description_width': 'initial'}
)

# Define your features (text_features, num_features, and cat_features)
text_features = ['video_title', 'video_description', 'video_tags']
num_features = ['video_duration', 'video_published_at', 'video_count',
                'mean_views', 'median_views']
cat_features = ['Action game', 'Action-adventure game', 'American football',
                'Association football', 'Baseball', 'Basketball', 'Boxing',
                'Business', 'Casual game', 'Christian music', 'Classical music',
                'Country music', 'Cricket', 'Electronic music', 'Entertainment',
                'Fashion', 'Film', 'Food', 'Golf', 'Health', 'Hip hop music',
                'Hobby', 'Humour', 'Ice hockey', 'Independent music', 'Jazz',
                'Knowledge', 'Lifestyle (sociology)', 'Military',
                'Mixed martial arts', 'Motorsport', 'Music', 'Music of Asia',
                'Music of Latin America', 'Music video game', 'Performing arts',
                'Pet', 'Physical fitness', 'Politics', 'Pop music',
                'Professional wrestling', 'Puzzle video game',
                'Racing video game', 'Reggae', 'Religion', 'Rhythm and blues',
                'Rock music', 'Role-playing video game', 'Simulation video game',
                'Society', 'Soul music', 'Sport', 'Sports game',
                'Strategy video game', 'Technology', 'Television program',
                'Tennis', 'Tourism', 'Vehicle', 'Video game culture']

# Create groups of features to organize into columns
features = text_features + num_features + cat_features

# # For the time being, let's remove numerical features. I'd prefer to try
# # making predictions without them, and, futher, duration isn't parse properly.
# features = text_features + cat_features

col_size = 15
feature_groups = [features[i:i+col_size] for i in range(0, len(features), col_size)]

feature_checks = {}  # Dictionary to store the checkboxes
grid = []  # List to hold columns

# Create checkboxes and organize them into a grid
for group in feature_groups:
    column_group = []
    for feature in group:
        feature_checks[feature] = widgets.Checkbox(description=feature,
                                                   value=False)
        column_group.append(feature_checks[feature])
    column = widgets.VBox(column_group)
    grid.append(column)

# Display the grid of checkboxes
grid = widgets.HBox(grid)

# Create buttons to toggle text, num, cat features and a random toggle button
text_button_on = widgets.Button(description="ON Text Features")
text_button_off = widgets.Button(description="OFF Text Features")
text_button_random = widgets.Button(description="Randomize Text")

num_button_on = widgets.Button(description="ON Num Features")
num_button_off = widgets.Button(description="OFF Num Features")
num_button_random = widgets.Button(description="Randomize Num")

cat_button_on = widgets.Button(description="ON Cat Features")
cat_button_off = widgets.Button(description="OFF Cat Features")
cat_button_random = widgets.Button(description="Randomize Cat")

random_button = widgets.Button(description="Party Mode")

# Define the toggle function for text features
def toggle_text_features_on(b):
    current_state = feature_checks[text_features[0]].value  # Get the current state of the first text feature
    for feature in text_features:
        feature_checks[feature].value = True  # Toggle all text features

def toggle_text_features_off(b):
    current_state = feature_checks[text_features[0]].value  # Get the current state of the first text feature
    for feature in text_features:
        feature_checks[feature].value = False  # Toggle all text features

def random_text_features(b):
    for feature in text_features:
        feature_checks[feature].value = random.choice([True, False])  # Randomly set each feature to True or False

# Define the toggle function for num features
def toggle_num_features_on(b):
    current_state = feature_checks[num_features[0]].value  # Get the current state of the first num feature
    for feature in num_features:
        feature_checks[feature].value = True

def toggle_num_features_off(b):
    current_state = feature_checks[num_features[0]].value  # Get the current state of the first num feature
    for feature in num_features:
        feature_checks[feature].value = False

def random_num_features(b):
    for feature in num_features:
        feature_checks[feature].value = random.choice([True, False])  # Randomly set each feature to True or False

# Define the toggle function for cat features
def toggle_cat_features_on(b):
    current_state = feature_checks[cat_features[0]].value  # Get the current state of the first cat feature
    for feature in cat_features:
        feature_checks[feature].value = True

def toggle_cat_features_off(b):
    current_state = feature_checks[cat_features[0]].value  # Get the current state of the first cat feature
    for feature in cat_features:
        feature_checks[feature].value = False

def random_cat_features(b):
    for feature in cat_features:
        feature_checks[feature].value = random.choice([True, False])  # Randomly set each feature to True or False

# Define the random toggle function
def random_toggle_features(b):
    random_text_features(b)
    random_num_features(b)
    random_cat_features(b)

# Attach the toggle functions to their respective buttons

text_button_on.on_click(toggle_text_features_on)
text_button_off.on_click(toggle_text_features_off)
text_button_random.on_click(random_text_features)
text_buttons = widgets.HBox([text_button_on, text_button_off, text_button_random])

num_button_on.on_click(toggle_num_features_on)
num_button_off.on_click(toggle_num_features_off)
num_button_random.on_click(random_num_features)
num_buttons = widgets.HBox([num_button_on, num_button_off, num_button_random])

cat_button_on.on_click(toggle_cat_features_on)
cat_button_off.on_click(toggle_cat_features_off)
cat_button_random.on_click(random_cat_features)
cat_buttons = widgets.HBox([cat_button_on, cat_button_off, cat_button_random])

random_button.on_click(random_toggle_features)

button_list = [
    widgets.Label(value=""),
    text_button_on,
    text_button_off,
    text_button_random,
    widgets.Label(value=""),
    num_button_on,
    num_button_off,
    num_button_random,
    widgets.Label(value=""),
    cat_button_on,
    cat_button_off,
    cat_button_random,
    widgets.Label(value=""),
    random_button
]

# Display the buttons and the grid side by side
button_box = widgets.VBox(button_list)
column_selection_layout = widgets.HBox([button_box, grid])

# Fractional Sliders
train_eval_split_slider = widgets.FloatSlider(
    value=.25,
    min=0.0,
    max=1.0,
    step=0.01,
    description='Eval Data Fraction of Total',
    disabled=False,
    continuous_update=False,
    orientation='horizontal',
    readout=True,
    layout=widgets.Layout(width='70%'),  # Set the width of the slider to 80%
    style={'description_width': 'initial'}
)

val_test_split_slider = widgets.FloatSlider(
    value=.5,
    min=0.0,
    max=1.0,
    step=0.01,
    description='Test Data Fraction of Eval',
    disabled=False,
    continuous_update=False,
    orientation='horizontal',
    readout=True,
    layout=widgets.Layout(width='70%'),  # Set the width of the slider to 80%
    style={'description_width': 'initial'}
)

# train bar
train_label = widgets.Label(value="Train",
                            layout=widgets.Layout(width='128px'))

train_racing_channel_toggle = widgets.ToggleButtons(
    options=['Racing Channels', 'All', 'No Racing Channels'],
    value=segment_value,
    description='Segment',
    style={'description_width': '128px'}
)

train_racing_video_toggle = widgets.ToggleButtons(
    options=['Racing Videos', 'All', 'No Racing Videos'],
    value=video_value,
    description='Observations',
    style={'description_width': '128px'}
)


train_viewership_threshold = widgets.IntText(
    value=threshold_value,
    description='Threshold',
    style={'description_width': '128px'}
)

train_widgets = [
    train_label,
    train_racing_channel_toggle,
    train_racing_video_toggle,
    train_viewership_threshold
]

train_bar = widgets.HBox(train_widgets)

# eval bar
eval_label = widgets.Label(value="Eval",
                           layout=widgets.Layout(width='128px'))

eval_racing_channel_toggle = widgets.ToggleButtons(
    options=['Racing Channels', 'All', 'No Racing Channels'],
    value=segment_value,
    description='Segment',
    style={'description_width': '128px'}
)

eval_racing_video_toggle = widgets.ToggleButtons(
    options=['Racing Videos', 'All', 'No Racing Videos'],
    value=video_value,
    description='Observations',
    style={'description_width': '128px'}
)

eval_viewership_threshold = widgets.IntText(
    value=threshold_value,
    description='Threshold',
    style={'description_width': '128px'}
)

eval_widgets = [
    eval_label,
    eval_racing_channel_toggle,
    eval_racing_video_toggle,
    eval_viewership_threshold
]

eval_bar = widgets.HBox(eval_widgets)

# threshold style
threshold_style = widgets.ToggleButtons(
    options=['Either', 'Both', 'Mean', 'Median'],
    description='Threshold Style',
    style={'description_width': 'initial'}
)

# Random State
random_state_input = widgets.IntText(
    value=42,
    description='Random State',
    style={'description_width': '128px'}
)

# Eval channel
eval_channel = widgets.Text(
    value='UCy2_huIuBrz6pBAd9kaB-gw',
    description='Eval Channel ID',
    disabled=False,
    style={'description_width': '128px'}
)


# Adding a spacer (a Box with some height) between grid layout and slider
spacer = widgets.Box(layout=widgets.Layout(height='32px', width='32px'))

# Show
display(spacer)  # This adds a space between the grid layout and slider
display(target_style)
display(spacer)  # This adds a space between the grid layout and slider
display(resample_mode)
display(spacer)
display(column_selection_layout)
display(spacer)  # This adds a space between the grid layout and slider
display(train_eval_split_slider)
display(spacer)  # This adds a space between the grid layout and slider
display(val_test_split_slider)
display(spacer)  # This adds a space between the grid layout and slider
display(train_bar)
display(spacer)  # This adds a space between the grid layout and slider
display(eval_bar)
display(spacer)  # This adds a space between the grid layout and slider
display(threshold_style)
display(spacer)  # This adds a space between the grid layout and slider
display(random_state_input)
display(spacer)  # This adds a space between the grid layout and slider
display(eval_channel)

Box(layout=Layout(height='32px', width='32px'))

ToggleButtons(description='Target Binary Threshold', options=('Median', 'Mean'), style=ToggleButtonsStyle(desc…

Box(layout=Layout(height='32px', width='32px'))

ToggleButtons(description='Resample Mode', options=(None, True, False), style=ToggleButtonsStyle(description_w…

Box(layout=Layout(height='32px', width='32px'))

...

ToggleButtons(description='Threshold Style', options=('Either', 'Both', 'Mean', 'Median'), style=ToggleButtons…

Box(layout=Layout(height='32px', width='32px'))

IntText(value=42, description='Random State', style=DescriptionStyle(description_width='128px'))

Box(layout=Layout(height='32px', width='32px'))

Text(value='UCy2_huIuBrz6pBAd9kaB-gw', description='Eval Channel ID', style=DescriptionStyle(description_width…

In [ ]:
 # @title Parameterized Preprocess

%%time

from sklearn.model_selection import train_test_split

import torch

print(f'Num GPUs: {torch.cuda.device_count()}')

import numpy as np

# try rolling encodings instead of categories

# Adjustable Inputs
input_parameters = {
    'train_eval_split': train_eval_split_slider.value,
    'val_test_split': val_test_split_slider.value,
    'train_viewership_threshold': train_viewership_threshold.value,
    'train_racing_channels': train_racing_channel_toggle.value,
    'eval_viewership_threshold': eval_viewership_threshold.value,
    'eval_racing_channels': eval_racing_channel_toggle.value,
    'target_column_type': target_style.value,
    'random_seed': random_state_input.value,
    'eval_channel_id': eval_channel.value,
    'resample_mode': resample_mode.value,
    'threshold_style': threshold_style.value,
}

df = full_df.copy()
df['target'] = df['video_view_count'] > df[f'{target_style.value.lower()}_views']


# Extract my data
chan_df = df[df['channel_id'] == eval_channel.value].copy()
df = df[df['channel_id'] != eval_channel.value].copy()


# Split by timestamp
df.sort_values('video_published_at', inplace=True)
train_df = df.iloc[:int(len(df) * (1 - input_parameters['train_eval_split']))]
eval_df = df.iloc[int(len(df) * (1 - input_parameters['train_eval_split'])):]
test_df = eval_df.iloc[:int(len(eval_df) * (1 - input_parameters['val_test_split']))]
val_df = eval_df.iloc[int(len(eval_df) * (1 - input_parameters['val_test_split'])):]


# Split by ID
train_ids, eval_ids = train_test_split(
    df['channel_id'].unique(), test_size=input_parameters['val_test_split'],
    random_state=input_parameters['random_seed'])

val_ids, test_ids = train_test_split(
    eval_ids, test_size=input_parameters['val_test_split'],
    random_state=input_parameters['random_seed'])

train_df = train_df[train_df['channel_id'].isin(train_ids)]
val_df = val_df[val_df['channel_id'].isin(val_ids)]
test_df = test_df[test_df['channel_id'].isin(test_ids)]


# Filter Racing Channels
def filter_racing_channels(data, segment, racing_threshold=.5):
  racing_column = [col for col in data.columns if 'Racing' in col][0]
  if segment == 'Racing Channels':
    data_ids = data[data[racing_column] > racing_threshold].index.tolist()
  elif segment == 'No Racing Channels':
    data_ids = data[data[racing_column] < racing_threshold].index.tolist()
  else:
    data_ids = data.index.tolist()
  return data.loc[data_ids]

racing_filtered_train = filter_racing_channels(
    train_df, train_racing_channel_toggle.value)

racing_filtered_val = filter_racing_channels(
    val_df, eval_racing_channel_toggle.value)

racing_filtered_test = filter_racing_channels(
    test_df, eval_racing_channel_toggle.value)


# Filter by Viewership
def filter_by_viewership(data, threshold, threshold_type):
  mean_filtered_data = data[data['mean_views'] >= threshold]
  median_filtered_data = data[data['median_views'] >= threshold]
  mean_ids = mean_filtered_data.index.tolist()
  median_ids = median_filtered_data.index.tolist()
  if threshold_type == 'Median':
    filtered_ids = list(set(median_ids))
  elif threshold_type == 'Mean':
    filtered_ids = list(set(mean_ids))
  elif threshold_type == 'Either':
    filtered_ids = list(set(mean_ids) | set(median_ids))
  elif threshold_type == 'Both':
    filtered_ids = list(set(mean_ids) & set(median_ids))
  filtered_data = data[data.index.isin(filtered_ids)]
  return filtered_data

viewership_filtered_train = filter_by_viewership(
    racing_filtered_train, input_parameters['train_viewership_threshold'],
    input_parameters['threshold_style'])

viewership_filtered_val = filter_by_viewership(
    racing_filtered_val, input_parameters['eval_viewership_threshold'],
    input_parameters['threshold_style'])

viewership_filtered_test = filter_by_viewership(
    racing_filtered_test, input_parameters['eval_viewership_threshold'],
    input_parameters['threshold_style'])


# Select columns
use_features = [feature for feature in features
                if feature_checks[feature].value]

keep_features = [ft for ft in use_features if ft in df.columns]

text_features = [ft for ft in ['video_title', 'video_description', 'video_tags']
                 if ft in keep_features]

train_data = viewership_filtered_train[keep_features + ['target']]
val_data = viewership_filtered_val[keep_features + ['target']]
test_data = viewership_filtered_test[keep_features + ['target']]


# Drop zero columns
train_data = train_data.loc[:, (train_data != 0).any(axis=0)]
val_data = val_data[train_data.columns]
test_data = test_data[train_data.columns]


# Drop NA
train_data = train_data.dropna(subset=['target'])
val_data = val_data.dropna(subset=['target'])
test_data = test_data.dropna(subset=['target'])


# Resample
def stratified_resample(training_data, testing_data, target, random_seed):

  # evaluation_data = pd.concat([validation_data, testing_data])
  evaluation_data = testing_data.copy()
  evaluation_value_counts = evaluation_data['target'].value_counts()
  evaluation_percentages = evaluation_value_counts / evaluation_data.shape[0]

  train_value_counts = training_data['target'].value_counts()
  train_percentages = train_value_counts / training_data.shape[0]

  not_target_count = train_value_counts[not target]
  not_target_pct = evaluation_percentages[not target]
  resampled_len = int(not_target_count / not_target_pct)
  target_resampled_count = resampled_len - not_target_count

  target_data = training_data[training_data['target'] == target]
  not_target_data = training_data[~training_data['target'] == target]

  train_target_pct= train_percentages[target]
  eval_target_pct = evaluation_percentages[target]

  if train_target_pct > eval_target_pct:

    target_resampled = target_data.sample(n=target_resampled_count,
                                         random_state=random_seed)

    training_data = pd.concat([target_resampled, not_target_data])

  elif train_target_pct < eval_target_pct:

    resample_multiple = target_resampled_count // target_data.shape[0]
    resample_remainder = target_resampled_count % target_data.shape[0]
    target_resampled = target_data.sample(n=resample_remainder,
                                          random_state=random_seed)
    resample_datas = [target_data] * resample_multiple + [target_resampled,
                                                            not_target_data]
    training_data = pd.concat(resample_datas)

  return training_data

if input_parameters['resample_mode'] is not None:
  train_data = stratified_resample(train_data, test_data,
                                   input_parameters['resample_mode'],
                                   input_parameters['random_seed'])
  val_data = stratified_resample(val_data, test_data,
                                 input_parameters['resample_mode'],
                                 input_parameters['random_seed'])


# Display pre-train metadata
train_size = train_data.shape[0]
val_size = val_data.shape[0]
test_size = test_data.shape[0]

train_true_pct = train_data[train_data['target'] == True].shape[0] / train_size
val_true_pct = val_data[val_data['target'] == True].shape[0] / val_size
test_true_pct = test_data[test_data['target'] == True].shape[0] / test_size

train_pct_total = train_size / (train_size + val_size + test_size)
val_pct_total = val_size / (train_size + val_size + test_size)
test_pct_total = test_size / (train_size + val_size + test_size)

print()
print(f'Train | True%: {round(train_true_pct, 3)} | %Total: {round(train_pct_total, 3)} | Shape: {train_data.shape}')
print(f'Val   | True%: {round(val_true_pct, 3)} | %Total: {round(val_pct_total, 3)} | Shape: {val_data.shape}')
print(f'Test  | True%: {round(test_true_pct, 3)} | %Total: {round(test_pct_total, 3)} | Shape: {test_data.shape}')
print()

In [ ]:
# @title Model Parameters

# Defaults
model_parameter_defaults = {
    'model_checkpoint_name': 'prajjwal1/bert-tiny',
    'validation_metric': 'roc_auc',
    'normalize_text': True,
    'learning_rate': 1.0e-4,
    'weight_decay': 1.0e-3,
    'lr_decay': .9,
    'lr_mult': 1,
    'val_checks_per_epoch': 1,
    'patience': 4,
    'fit_time_limit': 0,
    'per_gpu_batch_size': 1024,
    'batch_size': 1024,
}

custom_css = HTML("""
<style>
    .widget-label {
        font-size: 16px !important;  /* Increase font size for labels */
    }
    .widget-slider .widget-readout {
        font-size: 16px !important;  /* Increase font size for slider readout */
    }
</style>
""")

display(custom_css)

spacer = widgets.Box(layout=widgets.Layout(height='16px', width='32px'))

quality_level_selector = widgets.ToggleButtons(
    options=['Medium', 'High', 'Best'],
    description='Quality Peset',
    style={'description_width': 'initial'}
)

hpo_selector = widgets.Checkbox(
    value=False,
    description='HPO',
    disabled=False,
)

quality_level_bar = widgets.HBox([quality_level_selector, hpo_selector])

display(quality_level_bar)

display(spacer)

model_ckeckpoint_names = [
    'prajjwal1/bert-tiny',
    'albert-base-v2',
    'albert-large-v2',
    'albert-xlarge-v2',
    'albert-xxlarge-v2',
    'bert-base-uncased',
    'bert-base-cased',
    'bert-large-uncased',
    'bert-large-cased',
    'dslim/bert-base-NER',
    'google-bert/bert-large-cased-whole-word-masking',
    'google-bert/bert-large-uncased-whole-word-masking',
    'microsoft/deberta-v3-xsmall',
    'microsoft/deberta-v3-small',
    'microsoft/deberta-v3-base',
    'microsoft/deberta-v3-large',
    'distilbert/distilbert-base-uncased',
    'distilbert/distilbert-base-cased',
    'distilbert/distilroberta-base',
    'microsoft/mpnet-base',
    'google/electra-small-discriminator',
    'google/electra-base-discriminator',
    'google/electra-large-discriminator',
    'flboehm/youtube-bert',
]
# deberta, mdeberta

model_checkpoint_selector = widgets.Dropdown(
    options=model_ckeckpoint_names,
    value=model_parameter_defaults['model_checkpoint_name'],
    description='Model Name',
    disabled=False,
    style={'description_width': 'initial'}
)


# display(model_checkpoint_selector)

evaluation_metric_options = [
    'log_loss',
    "roc_auc",
    "accuracy",
    'f1',
    'f1_weighted',
    ]

metric_selector = widgets.Dropdown(
    options=evaluation_metric_options,
    value=model_parameter_defaults['validation_metric'],
    description='Evaluation Metric',
    disabled=False,
    style={'description_width': 'initial'}
)

# display(metric_selector)

normalize_text_input = widgets.Checkbox(
    value=model_parameter_defaults['normalize_text'],
    description='Normalize Text',
    disabled=False,
    style={'description_width': 'initial'}
)

model_bar = widgets.HBox([model_checkpoint_selector, spacer, metric_selector, spacer, normalize_text_input])

display(model_bar)

# display(normalize_text_input)

display(spacer)

learning_rate_slider = widgets.FloatLogSlider(
    value=model_parameter_defaults['learning_rate'],  # The default value
    base=10,  # Logarithmic base
    min=-6,  # log10(1.0e-6)
    max=-2,  # log10(1.0e-2)
    step=0.125,  # Adjust the step size if necessary
    description='Learning Rate',
    disabled=False,
    layout=widgets.Layout(width='70%'),
    style={'description_width': 'initial'},
    readout_format='.1e'
)

display(learning_rate_slider)

weight_decay_slider = widgets.FloatLogSlider(
    value=model_parameter_defaults['weight_decay'],  # The default value
    base=10,  # Logarithmic base
    min=-4,  # log10(1.0e-6)
    max=-2,  # log10(1.0e-2)
    step=0.125,  # Adjust the step size if necessary
    description='Weight Decay',
    disabled=False,
    layout=widgets.Layout(width='70%'),
    style={'description_width': 'initial'},
    readout_format='.1e'
)

display(weight_decay_slider)

learning_rate_decay_slider = widgets.FloatSlider(
    value=model_parameter_defaults['lr_decay'],
    min=.8,  # log10(1.0e-6)
    max=1,  # log10(1.0e-2)
    step=0.01,
    description='Learning Rate Decay',
    layout=widgets.Layout(width='70%'),
    style={'description_width': 'initial'},
)

display(learning_rate_decay_slider)

display(spacer)

learning_rate_multiplier_input = widgets.IntText(
    value=model_parameter_defaults['lr_mult'],
    description='Learning Rate Multiplier',
    style={'description_width': 'initial'}
)

display(learning_rate_multiplier_input)

validation_checks_per_epoch_input = widgets.IntText(
    value=model_parameter_defaults['val_checks_per_epoch'],
    description='Validation Checks / Epoch',
    style={'description_width': 'initial'}
)

display(validation_checks_per_epoch_input)

patience_input = widgets.IntText(
    value=model_parameter_defaults['patience'],
    description='Patience',
    style={'description_width': 'initial'}
)

display(patience_input)

time_limit_input = widgets.IntText(
    value=model_parameter_defaults['fit_time_limit'],
    description='Time Limit',
    style={'description_width': 'initial'}
)

display(time_limit_input)

display(spacer)

per_gpu_batch_size_input = widgets.IntText(
    value=model_parameter_defaults['per_gpu_batch_size'],
    description='Per GPU Batch Size',
    style={'description_width': 'initial'}
)

display(per_gpu_batch_size_input)

batch_size_input = widgets.IntText(
    value=model_parameter_defaults['batch_size'],
    description='Batch Size',
    style={'description_width': 'initial'}
)

display(batch_size_input)

In [ ]:
# @title Train Classifier RAM Fix

%%time

import torch

torch.set_float32_matmul_precision('medium')

hpo = '_hpo' if hpo_selector.value else ''

quality_preset = quality_level_selector.value.lower() + '_quality' + hpo
evaluation_metric = metric_selector.value
val_check_interval = 1 / validation_checks_per_epoch_input.value
time_limit = time_limit_input.value if time_limit_input.value > 0 else None
random_seed = input_parameters['random_seed']

# Adjustable Hypers
hyperparameters = {
    # Optimization Hyperparameters
    'optimization.learning_rate': learning_rate_slider.value,
    'optimization.weight_decay': weight_decay_slider.value,
    'optimization.lr_decay': learning_rate_decay_slider.value,
    'optimization.lr_mult': learning_rate_multiplier_input.value,
    'optimization.val_check_interval': val_check_interval,
    'optimization.patience': patience_input.value,
    # Data Hpyerparamete4rs
    'data.text.normalize_text': normalize_text_input.value,
    # Model hyperparameters
    'model.hf_text.checkpoint_name': model_checkpoint_selector.value,
    # Env Hyperparameters
    'env.per_gpu_batch_size': per_gpu_batch_size_input.value,
    'env.batch_size': batch_size_input.value,
}

from datetime import datetime

# get data sizes in GB
train_data_size = train_data.memory_usage(deep=True).sum() / (1024**3)
val_data_size = val_data.memory_usage(deep=True).sum() / (1024**3)
test_data_size = test_data.memory_usage(deep=True).sum() / (1024**3)

metadata = {
    'features': use_features,
    'train_rows': train_size,
    'train_true_pct': train_true_pct,
    'train_data_size': train_data_size,
    'val_rows': val_size,
    'val_true_pct': val_true_pct,
    'val_data_size': val_data_size,
    'test_rows': test_size,
    'test_true_pct': test_true_pct,
    'test_data_size': test_data_size,
    'timestamp': str(datetime.now()),
    'preset': quality_preset,
    'metric': evaluation_metric,
    'fit_time_limit': time_limit,
}

metadata_series = pd.Series(metadata, name='Metadata').to_frame()

metadata_series

import os
import json
import shutil

if os.path.exists('tmp/classifier'):
  shutil.rmtree('tmp/classifier')

os.makedirs('tmp/classifier')
os.makedirs('tmp/classifier/data')
os.makedirs('tmp/classifier/model')

train_data.to_csv('tmp/classifier/data/train.csv', index=False)
val_data.to_csv('tmp/classifier/data/val.csv', index=False)
test_data.to_csv('tmp/classifier/data/test.csv', index=False)

with open('tmp/classifier/data/input_parameters.json', 'w') as f:
    json.dump(input_parameters, f)

with open('tmp/classifier/data/metadata.json', 'w') as f:
    json.dump(metadata, f)

with open('tmp/classifier/data/hyperparameters.json', 'w') as f:
    json.dump(hyperparameters, f)

train_script = '''
from autogluon.multimodal import MultiModalPredictor
import json
from argparse import ArgumentParser
import pandas as pd

import torch

torch.set_float32_matmul_precision('medium')

from tqdm.notebook import tqdm
tqdm.pandas()

data_loc = 'tmp/classifier/data/'

# Read train, val, and test data
train_data = pd.read_csv(f'{data_loc}train.csv', engine='python')
train_data.dropna(subset=['target'], inplace=True)
print(train_data['target'].value_counts())
print(train_data['target'].value_counts().sum(), train_data.shape[0])
print()

val_data = pd.read_csv(f'{data_loc}val.csv', engine='python')
val_data.dropna(subset=['target'], inplace=True)
print(val_data['target'].value_counts())
print(val_data['target'].value_counts().sum(), val_data.shape[0])
print()

test_data = pd.read_csv(f'{data_loc}test.csv', engine='python')
test_data.dropna(subset=['target'], inplace=True)
print(test_data['target'].value_counts())
print(test_data['target'].value_counts().sum(), test_data.shape[0])
print()

with open(f'{data_loc}input_parameters.json', 'r') as f:
    input_parameters = json.load(f)

with open(f'{data_loc}metadata.json', 'r') as f:
    metadata = json.load(f)

with open(f'{data_loc}/hyperparameters.json', 'r') as f:
    hyperparameters = json.load(f)

# Set up argument parsing for additional command-line inputs
parser = ArgumentParser()
parser.add_argument('--quality_preset', type=str, help='Quality preset.')
parser.add_argument('--evaluation_metric', type=str, help='Evaluation metric.')
parser.add_argument('--time_limit', type=str, help='Time limit for training.')
parser.add_argument('--random_seed', type=int, help='Random seed.')

# Parse the arguments
args = parser.parse_args()

# Extract arguments from the parser
quality_preset = args.quality_preset
evaluation_metric = args.evaluation_metric
time_limit = args.time_limit
random_seed = args.random_seed

time_limit = None if time_limit == 'None' else int(time_limit)

print(pd.Series(hyperparameters, name='Hyperparameters').to_frame())

# Create the classifier using AutoGluon's MultiModalPredictor
classifier = MultiModalPredictor(
    label="target",  # Modify the label name if your target column differs
    problem_type="binary",
    presets=quality_preset,
    eval_metric=evaluation_metric,
    hyperparameters=hyperparameters,
    verbosity=4,
    validation_metric=evaluation_metric,
)

# Fit the classifier with train and validation data
classifier.fit(
    train_data=train_data,
    tuning_data=val_data,
    time_limit=time_limit,
    hyperparameters=hyperparameters,
    seed=random_seed,
    clean_ckpts=True
)

classifier.save('tmp/classifier/model')

evaluation_metrics = ['roc_auc', 'precision', 'recall', 'f1', 'average_precision',
                      'precision_weighted', 'recall_weighted', 'f1_weighted']

train_results = classifier.evaluate(train_data, metrics=evaluation_metrics)

val_results = classifier.evaluate(val_data, metrics=evaluation_metrics)

test_results = classifier.evaluate(test_data, metrics=evaluation_metrics)

results = [train_results, val_results, test_results]

results_by_partition_df = pd.DataFrame(results, index=['train', 'val', 'test'])

results_by_partition_df['len'] = [
  train_data.shape[0],
  val_data.shape[0],
  test_data.shape[0]
  ]

results_by_partition_df['true_pct'] = [
  train_data[train_data['target'] == True].shape[0] / train_data.shape[0],
  val_data[val_data['target'] == True].shape[0] / val_data.shape[0],
  test_data[test_data['target'] == True].shape[0] / test_data.shape[0]
  ]

results_by_partition_df['pct_of_total'] = [
  train_data.shape[0] / results_by_partition_df['len'].sum(),
  val_data.shape[0] / results_by_partition_df['len'].sum(),
  test_data.shape[0] / results_by_partition_df['len'].sum()
  ]

results_by_partition_df['partition'] = ['train', 'val', 'test']

results_by_partition_df.set_index('partition', inplace=True)

results_by_partition_df.to_csv(f'{data_loc}results_by_partition.csv', index=True)

summary = classifier.fit_summary()

run_data = {**test_results, **input_parameters, **metadata, **hyperparameters,
            **summary}

run_series = pd.Series(run_data)

results_df = run_series.to_frame().T.set_index('timestamp')

results_loc = '/content/drive/MyDrive/colab/banshee/results'

import os

if not os.path.exists(results_loc):
  os.makedirs(results_loc)

val_loc = f'{results_loc}/val_results.csv'

# Check if the CSV file already exists
if not os.path.isfile(val_loc):
    # File doesn't exist, so write the DataFrame with headers
    results_df.to_csv(val_loc, mode='w')
else:
    # File exists, so append the DataFrame without writing the header again
    results_df.to_csv(val_loc, mode='a', header=False)

'''

# Write the script to a file
with open('tmp/train_script.py', 'w') as f:
    f.write(train_script)

!python tmp/train_script.py --quality_preset {quality_preset} \
--evaluation_metric {evaluation_metric} \
--time_limit {time_limit} \
--random_seed {random_seed}

results_by_partition_df = pd.read_csv('tmp/classifier/data/results_by_partition.csv',
                                      index_col='partition')

results_by_partition_df.index = ['train', 'val', 'test']

results_by_partition_df

In [ ]:
#youtube-bert is best or tied for?

results_by_partition_df = pd.read_csv(
    'tmp/classifier/data/results_by_partition.csv', index_col='partition')

display(results_by_partition_df)

display(pd.Series({**input_parameters, **hyperparameters}))

In [ ]:
# evlaute precision with raised threshold
from autogluon.multimodal import MultiModalPredictor

classifier = MultiModalPredictor.load('tmp/classifier/model')

from sklearn.metrics import precision_score

prediction_df = test_data.copy()

prediction_df['prob'] = classifier.predict_proba(prediction_df)[True]

thresholds = [0 + i * .05 for i in range(20)]

for thr in thresholds:
  temp_df = prediction_df.copy()
  temp_df['pred'] = temp_df['prob'] > thr
  precision = precision_score(temp_df['target'], temp_df['pred'])
  print(f'Threshold: {thr}, Precision: {precision}')

prediction_df.sort_values('prob', ascending=False)

In [ ]:
# @title Utils

from IPython.display import display
import ipywidgets as widgets
import pandas as pd
import os
import torch

wipe_val_results_button = widgets.Button(description="Reset Val results")
show_val_results_button = widgets.Button(description="Show Val results")
show_size_results_button = widgets.Button(description="Show Size Results")
wipe_autogluon_models_button = widgets.Button(description="Wipe Autogluon Models")

# Define the toggle function for text features
def reset_val_results(b):
  val_loc = '/content/drive/MyDrive/colab/banshee/results/val_results.csv'
  if os.path.exists(val_loc):
    os.remove(val_loc)
    print('Val results wiped')

def show_val_results(b):
  val_loc = '/content/drive/MyDrive/colab/banshee/results/val_results.csv'
  if os.path.exists(val_loc):
    returned_results_df = pd.read_csv(val_loc, index_col='timestamp')
    display(returned_results_df.tail(5))

def show_size_results(b):
  val_loc = '/content/drive/MyDrive/colab/banshee/results/val_results.csv'
  if os.path.exists(val_loc):
    returned_results_df = pd.read_csv(val_loc, index_col='timestamp')
    keep_cols = ['model.hf_text.checkpoint_name', 'env.per_gpu_batch_size',
                 'env.batch_size', 'train_data_size', 'val_data_size',
                 'test_data_size', 'train_rows', 'val_rows', 'test_rows',
                 'train_true_pct', 'val_true_pct', 'test_true_pct',	'preset']
    display(returned_results_df[keep_cols].tail(5))

def wipe_autogluon_models(b):
  !rm -rf AutogluonModels
  print('Autogluon Models wiped')

wipe_val_results_button.on_click(reset_val_results)
show_val_results_button.on_click(show_val_results)
show_size_results_button.on_click(show_size_results)
wipe_autogluon_models_button.on_click(wipe_autogluon_models)

display(wipe_val_results_button)
display(show_val_results_button)
display(show_size_results_button)
display(wipe_autogluon_models_button)

### Evaluation

In [ ]:
# @title Channel Evaluations

from autogluon.multimodal import MultiModalPredictor

# load most recent checkpoint
classifier = MultiModalPredictor.load('tmp/classifier/model')

chan_data = chan_df.copy()
current_data = current_df.copy().drop('video_view_count', axis=1)

intersection_columns = set(chan_data.columns) & set(current_data.columns)
intersection_columns = list(intersection_columns)

chan_data.drop(columns=intersection_columns, inplace=True)

chan_data = chan_data.merge(current_data, left_on='channel_id', right_index=True)
chan_data['pred'] = classifier.predict_proba(chan_data)[True]

views_pred_corr = chan_data['target'].corr(chan_data['pred'])
print(f'Views Correlation: {views_pred_corr}')

chan_data[['video_title', 'video_view_count', 'target', 'pred']].sort_values('pred',
                                                            ascending=False)

In [ ]:
# @title Sales Evaluation

%%time

from datetime import datetime

nfs_sales = {
    'Need for Speed: Most Wanted (2005)': 17.8,
    'Need for Speed: Carbon': 15.6,
    'Need for Speed: Underground': 15,
    'Need for Speed: Underground 2': 11,
    'Need for Speed: ProStreet': 10.9,
    'Need for Speed: Undercover': 8.9,
    'Need for Speed: Hot Pursuit (2010)': 5,
    'Need for Speed Rivals': 4,
    'Need for Speed: Shift': 4
}

pred_rows = []
for key, value in nfs_sales.items():

  pred_row = current_data.copy()
  pred_row = pred_row.loc[eval_channel.value]

  pred_row['video_title'] = key
  pred_row['video_tags'] = key
  pred_row['video_description'] = key

  pred_row = pred_row.to_frame().T

  pred_row['video_published_at'] = datetime.now() + pd.Timedelta(hours=1)
  pred_row['video_published_at'] = pred_row['video_published_at'].dt.tz_localize(None)
  pred_row['video_published_at'] -= pd.Timestamp("1970-01-01")
  pred_row['video_published_at'] //= pd.Timedelta('1s')

  pred_row['video_duration'] = 1440
  pred_row['caption'] = True

  pred_rows.append(pred_row)

sales_df = pd.concat(pred_rows)
sales_df['sales'] = sales_df['video_title'].map(nfs_sales)
sales_df['prob'] = classifier.predict_proba(sales_df)[True]

sales_corr = sales_df['sales'].corr(sales_df['prob'])

print(f'Sales Correlation: {sales_corr}')

sales_df[['video_title', 'prob']].sort_values('prob', ascending=False)

In [ ]:
# @title Racing-Auto-Great Evaluation

current_chan = current_data.copy().loc[eval_channel.value].to_frame().T

racing_topics_list = ['Need for Speed: Carbon', 'Need for Speed: Most Wanted',
                 'Need for Speed: Underground', 'Need for Speed: Unbound',
                 'Need for Speed: The Run', 'Need for Speed: Underground 2',
                 'Need for Speed: Hot Pursuit', 'Need for Speed: ProStreet',
                 'Need for Speed: Shift', 'Midnight Club: Los Angeles',
                 'Burnout: Paradise', 'Dirt 3', 'Test Drive Unlimited',
                 'Forza Horizon 5', 'The Crew', 'Assetto Corsa',
                 'Project Gotham Racing 2', 'Gran Turismo 7', 'FlatOut',
                 'Forza Motorsport 8', 'Street Racing Syndicate']

racing_topic_current_data = [current_chan for _ in racing_topics_list]
racing_pred_feature_df = pd.concat(racing_topic_current_data)
racing_pred_feature_df['video_title'] = racing_topics_list
racing_pred_feature_df['video_description'] = racing_topics_list
racing_pred_feature_df['video_tags'] = racing_topics_list

racing_pred_feature_df['video_published_at'] = datetime.now()
racing_pred_feature_df['video_published_at'] = racing_pred_feature_df['video_published_at'].dt.tz_localize(None)
racing_pred_feature_df['video_published_at'] -= pd.Timestamp("1970-01-01")
racing_pred_feature_df['video_published_at'] //= pd.Timedelta('1s')

racing_pred_feature_df['video_duration'] = 1440
racing_pred_feature_df['video_caption'] = True
racing_pred_feature_df['score'] = 1

racing_pred_feature_df['prob'] = classifier.predict_proba(
    racing_pred_feature_df)[True]

racing_mean = racing_pred_feature_df['prob'].mean()
racing_median = racing_pred_feature_df['prob'].median()

auto_topics = ['Mad Max', 'Days Gone', 'Car Mechanic Simulator', 'CarX',
               'The Long Drive', 'Rocket League', 'Wreckfest', 'Split/Second',
               'Jalopy', 'Descenders', 'My Summer Car', 'F1', 'Driver',
               'Farming Simulator', 'BeamNG.drive', 'Trackmania']

auto_topic_current_data = [current_chan for _ in auto_topics]
auto_pred_feature_df = pd.concat(auto_topic_current_data)
auto_pred_feature_df['video_title'] = auto_topics
auto_pred_feature_df['video_description'] = auto_topics
auto_pred_feature_df['video_tags'] = auto_topics

auto_pred_feature_df['video_published_at'] = datetime.now()
auto_pred_feature_df['video_published_at'] = auto_pred_feature_df['video_published_at'].dt.tz_localize(None)
auto_pred_feature_df['video_published_at'] -= pd.Timestamp("1970-01-01")
auto_pred_feature_df['video_published_at'] //= pd.Timedelta('1s')

auto_pred_feature_df['video_duration'] = 1440
auto_pred_feature_df['video_caption'] = True
auto_pred_feature_df['score'] = .5

auto_pred_feature_df['prob'] = classifier.predict_proba(
    auto_pred_feature_df)[True]

auto_mean = auto_pred_feature_df['prob'].mean()
auto_median = auto_pred_feature_df['prob'].median()

great_topics = ['Elden Ring', 'Hades', 'Disco Elysium', 'God of War',
                'Red Dead Redemption 2', 'Fortnite', 'Stardew Valley',
                'The Legend of Zelda: Breath of the Wild', 'Overwatch',
                'Inside', 'The Witcher 3: Wild Hunt', 'Bloodborne',
                'Destiny', 'Mario Kart 8', 'Papers, Please', 'Left 4 Dead',
                'Grand Theft Auto V', 'The Last of Us', 'Batman: Arkham City',
                'The Elder Scrolls V: Skyrim', 'Dark Souls', 'Portal', 'Halo 3',
                'BioShock', 'Wii Sports', 'Resident Evil 4', 'Kingdom Hearts']

great_topic_current_data = [current_chan for _ in great_topics]
great_pred_feature_df = pd.concat(great_topic_current_data)
great_pred_feature_df['video_title'] = great_topics
great_pred_feature_df['video_description'] = great_topics
great_pred_feature_df['video_tags'] = great_topics

great_pred_feature_df['video_published_at'] = datetime.now()
great_pred_feature_df['video_published_at'] = great_pred_feature_df['video_published_at'].dt.tz_localize(None)
great_pred_feature_df['video_published_at'] -= pd.Timestamp("1970-01-01")
great_pred_feature_df['video_published_at'] //= pd.Timedelta('1s')

great_pred_feature_df['video_duration'] = 1440
great_pred_feature_df['video_caption'] = True
great_pred_feature_df['score'] = 0

great_pred_feature_df['prob'] = classifier.predict_proba(
    great_pred_feature_df)[True]

great_mean = great_pred_feature_df['prob'].mean()
great_median = great_pred_feature_df['prob'].median()

pred_df = pd.concat([racing_pred_feature_df, auto_pred_feature_df,
                     great_pred_feature_df])

rag_corr = pred_df['score'].corr(pred_df['prob'])

print(f'Correlation: {rag_corr}')

gb_prob = pred_df.groupby('score')['prob']
means = gb_prob.mean()
medians = gb_prob.median()

central_df = pd.concat([means, medians], axis=1)
central_df.columns = ['mean', 'median']

cat_map = {
    1: 'Racing',
    .5: 'Auto',
    0: 'Great'
}

central_df.index = central_df.index.map(cat_map)

print(central_df.sort_values('median', ascending=False))

keep_features = [ft for ft in features if ft in pred_df.columns]

pred_df[keep_features + ['score', 'prob']].sort_values('prob', ascending=False).reset_index(drop=True)

In [ ]:
corr_data = {
    'Eval Views-Pred Corr': views_pred_corr,
    'Sales Corr': sales_corr,
    'Racing-Auto-Great Corr': rag_corr,
}

corr_df = pd.Series(corr_data, name='Correlations').to_frame()

corr_df

In [ ]:
# @title My Games Evaluation

my_games = [
  "112 Operator", "3DMark", "911 Operator", "ATOM RPG",
  "Age of Empires II: Definitive Edition", "Amnesia: A Machine for Pigs",
  "Amnesia: Rebirth", "Amnesia: The Dark Descent", "Anno 1800", "Apex Legends",
  "Assassin's Creed IV Black Flag", "Atomic Heart",
  "Attack on Titan 2 - A.O.T.2", "BONEWORKS", "Back 4 Blood", "Balatro",
  "BattleBit Remastered", "Beat Saber", "Betrayal At Club Low",
  "Beyond: Two Souls", "BioShock", "BioShock 2", "BioShock 2 Remastered",
  "BioShock Infinite", "BioShock Remastered", "Blair Witch",
  "Bomb Rush Cyberfunk", "Book of Demons", "Borderlands 2", "Borderlands 3",
  "Borderlands GOTY", "Borderlands GOTY Enhanced",
  "Borderlands: The Pre-Sequel", "Braveland Pirate", "Brawlhalla", "Broken Age",
  "Brothers - A Tale of Two Sons", "Bully: Scholarship Edition",
  "Burnout™ Paradise Remastered", "Bus Simulator 23", "CODE VEIN",
  "Call of Duty 4: Modern Warfare (2007)", "Call of Duty: Black Ops",
  "Call of Duty: Black Ops - Multiplayer", "Car Mechanic Simulator 2014",
  "Car Mechanic Simulator 2018", "Chernobylite Complete Edition",
  "Cities: Skylines", "Company of Heroes 2", "Corridor Z", "Counter-Strike 2",
  "Creatures Docking Station", "Cruelty Squad", "Crusader Kings III",
  "Crying Suns", "Cultist Simulator", "Cyberpunk 2077",
  "DARK SOULS™: REMASTERED", "DEATHLOOP", "DREAM LOGIC", "DREDGE",
  "Darkest Dungeon®", "Darkest Dungeon® II", "Dawn of Man", "Days Gone",
  "Dead End Road", "Dead Rising", "Dear Esther: Landmark Edition",
  "Democracy 3", "Democracy 3 Africa", "Destiny 2", "Detroit: Become Human",
  "Disco Elysium", "Dragon's Dogma: Dark Arisen", "Draw Slasher", "Drawful 2",
  "Driftland: The Magic Revival", "ELDEN RING", "Enderal: Forgotten Stories",
  "Escape the Backrooms", "Europa Universalis IV", "Expeditions: Viking",
  "FAITH", "FINAL FANTASY VII", "FINAL FANTASY VII REMAKE INTERGRADE",
  "FINAL FANTASY VIII - REMASTERED", "FINAL FANTASY X/X-2 HD Remaster",
  "FIVE NIGHTS AT FREDDY'S: HELP WANTED", "Fable Anniversary", "Fall Guys",
  "Fallout", "Fallout 2", "Fallout 3 - Game of the Year Edition", "Fallout 4",
  "Fallout 76", "Fallout Tactics", "Fallout: New Vegas", "Firewatch",
  "Frostpunk", "GameGuru Classic", "Getting Over It with Bennett Foddy",
  "Goat Simulator", "Goat Simulator 3", "Gone Home", "Google Earth VR",
  "Gorogoa", "Grand Theft Auto IV: The Complete Edition", "Grand Theft Auto V",
  "GridIron", "Hades", "Halo Infinite", "Heavy Rain", "Hollow Knight",
  "Horizon Zero Dawn™ Complete Edition", "INMOST", "INSIDE", "Inscryption",
  "Inside the Backrooms", "It Takes Two", "Jalopy", "Jet Set Radio",
  "Jurassic World Evolution", "Jurassic World Evolution 2",
  "KINGDOM HEARTS -HD 1.5+2.5 ReMIX-", "Kenshi", "Kentucky Route Zero",
  "Kerbal Space Program", "Kingdom Come: Deliverance", "Kingdom Two Crowns",
  "Kingdoms of Amalur: Re-Reckoning", "KinitoPET", "LIMBO", "Last Train Home",
  "Layers of Fear (2016)", "Learning Factory", "Left 4 Dead", "Left 4 Dead 2",
  "Lethal Company", "Little Brother Jim", "Little Nightmares",
  "Little Nightmares II", "Lust for Darkness", "Lust from Beyond: M Edition",
  "Mad Max", "Mafia: Definitive Edition", "MapleStory", "Marbles on Stream",
  "Metal: Hellsinger", "Metro Exodus", "Metro Exodus Enhanced Edition",
  "Middle-earth™: Shadow of War™", "Monaco", "Moon Hunters",
  "Motorcycle Mechanic Simulator 2021", "Motorsport Manager",
  "Mount & Blade II: Bannerlord", "Mount & Blade: Warband", "NecroWorm",
  "Need for Speed™ Hot Pursuit Remastered", "Need for Speed™ Unbound",
  "Neon Hardcore", "Neon White", "Neverout", "Nex Machina",
  "Ni no Kuni™ II: Revenant Kingdom", "NieR Replicant ver.1.22474487139...",
  "NieR:Automata™", "OCTOPATH TRAVELER", "Observer", "OlliOlli World",
  "Orbital Racer", "Out of Reach: Treasure Royale", "PGA TOUR 2K21",
  "Pacific Drive", "Palworld", "Papers, Please", "Pathologic 2", "Pathway",
  "Pawnbarian", "Penumbra: Black Plague", "Penumbra: Overture",
  "Penumbra: Requiem", "Pikuniku", "Planet Coaster", "Planet Zoo",
  "Political Animals", "Pony Island", "Popup Dungeon", "Quantum Break",
  "RPG Maker VX", "Red Dead Redemption 2", "Resident Evil 7 Biohazard",
  "Resident Evil 7 Teaser: Beginning Hour", "Resident Evil Re:Verse",
  "Resident Evil Village", "Ring of Pain", "Rival Stars Horse Racing",
  "Rocket League", "Rocksmith® 2014 Edition - Remastered", "Rollerdrome",
  "Ryse: Son of Rome", "S.T.A.L.K.E.R.: Shadow of Chernobyl",
  "SEGA Mega Drive & Genesis Classics", "SOMA", "Saints Row 2", "Satisfactory",
  "Sea of Thieves", "Sekiro™: Shadows Die Twice", "Shadow of the Tomb Raider",
  "Shadowhand", "Sid Meier's Civilization VI", "Sid Meier's Pirates!",
  "Simply Chess", "Skullgirls 2nd Encore", "Slay the Princess",
  "Slay the Spire", "Sleeping Dogs: Definitive Edition", "SnowRunner",
  "Sonic & All-Stars Racing Transformed Collection", "Sonic Adventure DX",
  "Sonic Adventure™ 2 ", "South Park The Fractured But Whole",
  "South Park™: The Stick of Truth™", "Split/Second", "Splitgate", "Spore",
  "Spyro™ Reignited Trilogy", "Starbound", "Starbound - Unstable",
  "Stardew Valley", "Steelrising", "Subnautica", "Sunset Overdrive",
  "Super Monkey Ball Banana Mania", "Supraland", "System Shock: Classic",
  "System Shock: Enhanced Edition", "TUNIC", "Tannenberg", "Tempest",
  "The Amazing American Circus", "The Beginner's Guide",
  "The Complex: Found Footage", "The Crew", "The Crew 2",
  "The Elder Scrolls III: Morrowind", "The Elder Scrolls IV: Oblivion ",
  "The Elder Scrolls Online", "The Elder Scrolls V: Skyrim", "The Forest",
  "The Hex", "The Last of Us™ Part I", "The Long Dark", "The Medium",
  "The Sims™ 4", "The Stanley Parable", "The Stanley Parable: Ultra Deluxe",
  "The Talos Principle", "The Turing Test", "The Unfinished Swan",
  "The Walking Dead: The Telltale Definitive Series",
  "The Witcher 2: Assassins of Kings Enhanced Edition",
  "The Witcher 3: Wild Hunt", "The Witcher: Enhanced Edition",
  "This War of Mine", "Thrillville: Off the Rails", "Tomb Raider",
  "Tooth and Tail", "Trackmania", "Tropico 6", "Twelve Minutes",
  "UNCHARTED™: Legacy of Thieves Collection", "VRChat", "VTube Studio",
  "Vagante", "Vampyr", "Verdun", "WARSAW RISING: City of Heroes",
  "WORLD OF HORROR", "Wargroove", "Way of the Hunter", "We. The Revolution",
  "West of Dead", "What Remains of Edith Finch", "When the Darkness comes",
  "Windward", "Wizard of Legend", "Workers & Resources: Soviet Republic",
  "Wreckfest", "Wreckfest Throw-A-Santa + Sneak Peek 2.0",
]

my_games = sorted(my_games)

from itertools import product

suffixes = ['', 'Gameplay', 'Walkthrough', 'Review', 'Retrospective',
            'Analysis', 'Critique']

pairs = list(product(my_games, suffixes))

pair_df = pd.DataFrame(pairs, columns=['game', 'suffix'])

pair_df['video_title'] = pair_df['game'] + ' ' + pair_df['suffix']

titles = pair_df['video_title'].values
games = pair_df['game'].values
suffixes = pair_df['suffix'].values

current_chan = current_data.copy().loc[eval_channel.value].to_frame().T

pred_rows = [current_chan for _ in titles]

pred_feature_df = pd.concat(pred_rows)

pred_feature_df['video_title'] = titles
pred_feature_df['video_tags'] = titles
pred_feature_df['video_description'] = titles
pred_feature_df['game'] = games
pred_feature_df['suffix'] = suffixes

pred_feature_df['video_duration'] = 1440

pred_feature_df['video_published_at'] = datetime.now()
pred_feature_df['video_published_at'] = pred_feature_df['video_published_at'].dt.tz_localize(None)
pred_feature_df['video_published_at'] -= pd.Timestamp("1970-01-01")
pred_feature_df['video_published_at'] //= pd.Timedelta('1s')

pred_feature_df['prob'] = classifier.predict_proba(pred_feature_df)[True]

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 512)

game_gb = pred_feature_df.groupby('game')['prob'].describe().drop('count', axis=1)

print(game_gb.sort_values('50%', ascending=False))

suffix_gb = pred_feature_df.groupby('suffix')['prob'].describe().drop('count', axis=1)

print(suffix_gb.sort_values('50%', ascending=False))

In [ ]:
# @title Save Model

date_str = datetime.now().strftime('%Y_%m_%d')
time_str = datetime.now().strftime('%H_%M_%S')

save_dir = f'/content/drive/MyDrive/colab/banshee/models/{date_str}/{time_str}'

if not os.path.exists(save_dir):
  os.makedirs(save_dir)

classifier.save(save_dir, standalone=True)

new_predictor = classifier.load(save_dir)

# WIP

In [ ]:
# @title ZSC

from transformers import pipeline

clasifier = pipeline("zero-shot-classification",
                     model='facebook/bart-large-mnli')
                    #  model='MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7')
                      # model='valhalla/distilbart-mnli-12-1')

titles = [
    "Need for Speed: Underground",
    "Need for Speed: Underground 2",
    "Need for Speed: Most Wanted",
    "Need for Speed: Carbon",
    "Need for Speed: ProStreet",
    "Need for Speed: Undercover",
    "Need for Speed: Shift",
    "Need for Speed: The Run",
    "Need for Speed Rivals",
    "Need for Speed Payback",
    "Need for Speed No Limits",
    "Need for Speed: Hot Pursuit",
    "Need for Speed Heat",
    "Need for Speed Unbound",
    'Forza Horizon',
    'Forza Horizon 2',
    'Forza Horizon 3',
    'Forza Horizon 4',
    'Forza Horizon 5',
    'Test Drive Unlimited',
    'Gran Turismo 7',
    'Gran Turismo 8',
    ]

candidate_labels = ['Need for Speed', 'Forza', 'Gran Turismo', 'Driver', 'Burnout', 'Test Drive']

classifications = []

for title in titles:
  classification = clasifier(title, candidate_labels)

  sequence = classification['sequence']
  labels = classification['labels']
  scores = classification['scores']

  label_score_map = dict(zip(labels, scores))

  high_score = label_score_map['Need for Speed']

  class_data = [sequence, high_score]
  classifications.append(class_data)

classification_df = pd.DataFrame(classifications, columns=['title', 'Need for Speed'])
classification_df

## Embed

In [ ]:
%%time

# mount drive if unmounted
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd

from sqlalchemy import create_engine
from tqdm.notebook import tqdm

tqdm.pandas()

# connect to the database
db_loc = f'sqlite:///drive/MyDrive/colab/banshee/data/nfs_m.db'
engine = create_engine(db_loc)

# read the data
vid_stats_df = pd.read_sql_table('video_statistics', engine)

chan_stats_df = pd.read_sql_table('channel_statistics', engine)

In [ ]:
# cosine similarity

from sklearn.metrics.pairwise import cosine_similarity, euclidean_distances
from sklearn.preprocessing import StandardScaler


def filter_similarity(data, threshold=.5, my_id='UCy2_huIuBrz6pBAd9kaB-gw'):

  data_df = data.copy()

  data_df['video_published_at'] = pd.to_datetime(data_df['video_published_at'])
  data_df['video_published_at'] = data_df['video_published_at'].dt.tz_localize(None)

  to_int_cols = ['video_published_at', 'video_duration', 'video_view_count']

  data_df = data_df[['channel_id'] + to_int_cols]

  for col in to_int_cols:
    data_df[col] = data_df[col].astype(int)

  groups = data_df.groupby('channel_id').mean(numeric_only=True)
  groups['count'] = data_df.groupby('channel_id').size()

  video_published_at_scaler = StandardScaler()
  video_duration_scaler = StandardScaler()
  video_view_count_scaler = StandardScaler()
  count_scaler = StandardScaler()

  groups['video_published_at'] = video_published_at_scaler.fit_transform(groups[['video_published_at']])
  groups['video_duration'] = video_duration_scaler.fit_transform(groups[['video_duration']])
  groups['video_view_count'] = video_view_count_scaler.fit_transform(groups[['video_view_count']])
  groups['count'] = count_scaler.fit_transform(groups[['count']])

  my_group = groups.loc[my_id].values.reshape(1, -1)

  groups['similarity'] = cosine_similarity(groups.values, my_group).flatten()
  groups['similarity'] += 1
  groups['similarity'] /= 2

  groups = groups[groups['similarity'] > threshold]

  return groups

prepartition(vid_stats_df)

In [ ]:
# @title Read Data (Embed)

%%time


from sqlalchemy import create_engine

from tqdm.notebook import tqdm

tqdm.pandas()


vid_stats_dfs = []

data_names = [
    'nfs_m.db',
    # 'nfs.db',
    # 'video_statistics.csv'
    ]

pbar_len = (len(data_names) + 1) * 30
pbar = tqdm(total=pbar_len)

for data_name in data_names:

  if 'db' in data_name:

    # connect to the database
    db_loc = f'sqlite:///drive/MyDrive/colab/banshee/data/{data_name}'
    engine = create_engine(db_loc)

    # read the data
    vid_stats_df = pd.read_sql_table('video_statistics', engine)

    pbar.update(20)

  elif 'csv' in data_name:

    vid_stats_df = pd.read_csv(f'drive/MyDrive/colab/banshee/data/{data_name}',
                               low_memory=False)
    pbar.update(20)

  vid_stats_rename_map = {
      'video_published_at': 'timestamp',
      'video_description': 'description',
      'video_title': 'title',
      'video_tags': 'tags',
      'channel_id': 'channel',
      'video_duration': 'duration',
      'video_topic_categories': 'video_topics',
      'channel_topic_categories': 'channel_topics',
      'video_caption': 'caption',
      'video_view_count': 'views',
  }
  # preproc
  vid_stats_df.rename(columns=vid_stats_rename_map, inplace=True)

  vid_stats_drop_cols = ['video_like_count', 'video_comment_count',
                        'video_licensed_content']

  vid_stats_df.drop(columns=vid_stats_drop_cols, inplace=True)

  vid_stats_df.set_index('video_id', inplace=True)

  # Fill NaN values with 0 (or any other default value)
  vid_stats_df.dropna(subset=['views'], inplace=True)

  # Now convert the column to int
  vid_stats_df['views'] = vid_stats_df['views'].astype(int)

  vid_stats_df.sort_values('timestamp', inplace=True)

  vid_stats_df['timestamp'] = pd.to_datetime(vid_stats_df['timestamp'])
  vid_stats_df['timestamp'] = vid_stats_df['timestamp'].dt.tz_localize(None)
  vid_stats_df['timestamp'] -= pd.Timestamp("1970-01-01")
  vid_stats_df['timestamp'] //= pd.Timedelta('1s')

  vid_stats_df = vid_stats_df[~vid_stats_df.index.duplicated(keep='first')]

  vid_stats_df['source'] = data_name

  vid_stats_dfs.append(vid_stats_df)

  pbar.update(10)

vid_stats_df = pd.concat(vid_stats_dfs)

vid_stats_df = vid_stats_df[~vid_stats_df.index.duplicated(keep='first')]

vid_stats_df.drop_duplicates(inplace=True)

vid_stats_df.head()

In [ ]:
import pandas as pd
import itertools

tqdm.pandas()

# Assuming vid_stats_df is your original DataFrame
temp_df = vid_stats_df.copy()

# Define a function to compute 'previous_titles' for each group
def compute_previous_titles(group):
    titles = group['title'].astype(str).tolist()
    # Use itertools.accumulate to efficiently compute cumulative concatenations
    accumulated_titles = list(itertools.accumulate(titles[:-1], lambda x, y: f"{x} {y}"))
    # Prepend an empty string to align with the group index
    group['previous_titles'] = [''] + accumulated_titles
    return group

# Apply the function to each group
temp_df = temp_df.groupby('channel', group_keys=False).progress_apply(compute_previous_titles)

print(temp_df.head())


In [ ]:
from transformers import BertTokenizer

# Load the tokenizer for BERT-tiny
tokenizer = BertTokenizer.from_pretrained('prajjwal1/bert-tiny')

# Tokenize your dataset
def tokenize_function(texts):
    return tokenizer(texts, padding='max_length', truncation=True, max_length=128)

# Tokenize the titles from your train_df
train_encodings = tokenize_function(train_df['title'].tolist())

class TitleDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        # Convert labels to Float tensor
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.float)
        return item

    def __len__(self):
        return len(self.labels)

# Prepare labels and dataset
train_labels = train_df['label'].astype(float).tolist()  # Convert to float
train_dataset = TitleDataset(train_encodings, train_labels)


In [ ]:
from transformers import BertForSequenceClassification, Trainer, TrainingArguments
import torch

# Load the BERT-tiny model for binary classification (num_labels=1)
model = BertForSequenceClassification.from_pretrained('prajjwal1/bert-tiny', num_labels=1)

# Set up training arguments
training_args = TrainingArguments(
    output_dir='./results',               # directory for model checkpoints
    evaluation_strategy="epoch",          # evaluate at the end of each epoch
    learning_rate=2e-5,                   # learning rate
    per_device_train_batch_size=16,       # training batch size
    per_device_eval_batch_size=16,        # evaluation batch size
    num_train_epochs=3,                   # number of epochs
    weight_decay=0.01,                    # weight decay
    logging_dir='./logs',                 # directory for logs
    logging_steps=10,
)

# Initialize the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
)

# Fine-tune the model
trainer.train()

# Now that training is done, let's get embeddings
class BertTinyEmbeddingExtractor(torch.nn.Module):
    def __init__(self, model):
        super(BertTinyEmbeddingExtractor, self).__init__()
        # We want the BERT part only (not the classification head)
        self.bert = model.bert

    def forward(self, input_ids, attention_mask=None):
        # Pass inputs through the BERT model
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        # Retrieve the last hidden state embeddings (or pooled output)
        last_hidden_state = outputs.last_hidden_state  # Shape: (batch_size, sequence_length, hidden_size)
        pooled_output = outputs.pooler_output  # Shape: (batch_size, hidden_size) -> typically used for classification
        return last_hidden_state, pooled_output

# Initialize the embedding extractor
embedding_model = BertTinyEmbeddingExtractor(model)

# Example to get embeddings for some inputs
def get_embeddings(texts, tokenizer, embedding_model):
    # Tokenize the texts
    encodings = tokenizer(texts, return_tensors="pt", padding=True, truncation=True, max_length=128)

    # Extract embeddings
    with torch.no_grad():
        last_hidden_state, pooled_output = embedding_model(input_ids=encodings['input_ids'], attention_mask=encodings['attention_mask'])

    return last_hidden_state, pooled_output  # last_hidden_state gives token-level embeddings, pooled_output gives sentence-level embeddings

# Example usage with your data
texts = ["Example sentence 1", "Another example"]
last_hidden_state, pooled_output = get_embeddings(texts, tokenizer, embedding_model)

print("Token-level embeddings shape:", last_hidden_state.shape)  # Shape: (batch_size, sequence_length, hidden_size)
print("Sentence-level embeddings shape:", pooled_output.shape)   # Shape: (batch_size, hidden_size)


In [ ]:
# Extract embeddings for the 'title' column
texts = train_df['title'].tolist()
last_hidden_state, pooled_output = get_embeddings(texts, tokenizer, embedding_model)

# `pooled_output` now contains the sentence-level embeddings for each title
print("Pooled embeddings shape:", pooled_output.shape)
